## Reading Corpus.

### Notes on this corpus

- Contains **100 documents**
- Covers multiple topics:
    - machine learning
    - information retrieval
    - NLP
    - search systems
    - clustering, regression, etc.

- Designed to:
    - test **term frequency effects**
    - observe **IDF differences**
    - evaluate **ranking behavior**


In [2]:
with open("data/bm25_corpus.txt") as f:
    corpus = f.read()
    
    
corpus

'This document covers search engines with detailed insights. It provides\ncontext, concepts, and key ideas related to search engines and how they\nare applied in realistic scenarios. This document discusses regression\nanalysis in modern systems. It provides context, concepts, and key ideas\nrelated to regression analysis and how they are applied in realistic\nscenarios. This document covers natural language processing in modern\nsystems. It provides context, concepts, and key ideas related to natural\nlanguage processing and how they are applied in realistic scenarios.\nThis document explores clustering algorithms in modern systems. It\nprovides context, concepts, and key ideas related to clustering\nalgorithms and how they are applied in realistic scenarios. This\ndocument describes recommendation systems in modern systems. It provides\ncontext, concepts, and key ideas related to recommendation systems and\nhow they are applied in realistic scenarios. This document describes\nsearch 

### Cleaning corpus.
- Removing line break
- removing commons
- Break on basis of sentence end.
- convert to documents

In [3]:
corpus = corpus.replace("\n", " ").replace(",", "")
documents = corpus.split(".")

In [4]:
len(documents)

201

### Clean Documents
- Remove leading and tailing from documents

In [5]:
documents = [doc.strip() for doc in documents]

In [6]:
## Generate Vocab

N = len(documents)

vocab = set()
for doc in documents:
    for word in doc.split():
        vocab.add(word)
        
        
len(vocab)

58

In [7]:
# Simple tokenizer

def tokenize(document):
    return document.split(' ')


In [8]:
# Average Document length
avg_dl = round(sum(len(tokenize(doc)) for doc in documents) / N, 5)
avg_dl

13.69154

In [9]:
# Compute TF


def compute_tf(word: str, doc: str):
    word_count = doc.split().count(word)
    total_words = len(doc.split())

    tf = round(word_count / total_words, 5)

    return word_count, tf

In [10]:
# compute idf
import math


def compute_idf(word: str, documents: list, k: float = 0.5):
    df = 0
    for doc in documents:
        if word in doc.split():
            df += 1
    numerator = N - df + k
    denominator = df + k

    idf = round(math.log((numerator / denominator) + 1), 5)
    
    return idf, df

In [11]:
doc_frequency = {word: compute_idf(word, documents) for word in vocab}
doc_frequency

{'real': (2.28784, 20),
 'detailed': (2.95689, 10),
 'applied': (0.69811, 100),
 'in': (0.3702, 139),
 'applications': (2.28784, 20),
 'perspective': (2.95689, 10),
 'machine': (2.3905, 18),
 'context': (0.69811, 100),
 'transformer': (2.28784, 20),
 'provides': (0.69811, 100),
 'ideas': (0.69811, 100),
 'are': (0.69811, 100),
 'document': (0.69811, 100),
 'analyzes': (3.05698, 9),
 'algorithms': (2.63412, 14),
 'world': (2.28784, 20),
 'explains': (2.33785, 19),
 'regression': (2.03112, 26),
 'analysis': (2.03112, 26),
 'deep': (2.3905, 18),
 'retrieval': (2.95689, 10),
 'recommendation': (2.03112, 26),
 'a': (2.95689, 10),
 'It': (0.69811, 100),
 'beginners': (1.92388, 29),
 'systems': (1.53551, 43),
 'search': (2.50491, 16),
 'engines': (2.50491, 16),
 'data': (2.28784, 20),
 'covers': (2.50491, 16),
 'how': (0.69811, 100),
 'related': (0.69811, 100),
 'for': (1.92388, 29),
 'discusses': (2.19475, 22),
 'they': (0.69811, 100),
 'processing': (1.82703, 32),
 'information': (2.95689, 

In [12]:
# Document Term Frequency
term_frequency = {}

for index, doc in enumerate(documents):
    term_frequency[index] = {}
    for word in doc.split():
        term_frequency[index][word] = compute_tf(word, doc)
        
term_frequency

{0: {'This': (1, 0.125),
  'document': (1, 0.125),
  'covers': (1, 0.125),
  'search': (1, 0.125),
  'engines': (1, 0.125),
  'with': (1, 0.125),
  'detailed': (1, 0.125),
  'insights': (1, 0.125)},
 1: {'It': (1, 0.05263),
  'provides': (1, 0.05263),
  'context': (1, 0.05263),
  'concepts': (1, 0.05263),
  'and': (2, 0.10526),
  'key': (1, 0.05263),
  'ideas': (1, 0.05263),
  'related': (1, 0.05263),
  'to': (1, 0.05263),
  'search': (1, 0.05263),
  'engines': (1, 0.05263),
  'how': (1, 0.05263),
  'they': (1, 0.05263),
  'are': (1, 0.05263),
  'applied': (1, 0.05263),
  'in': (1, 0.05263),
  'realistic': (1, 0.05263),
  'scenarios': (1, 0.05263)},
 2: {'This': (1, 0.125),
  'document': (1, 0.125),
  'discusses': (1, 0.125),
  'regression': (1, 0.125),
  'analysis': (1, 0.125),
  'in': (1, 0.125),
  'modern': (1, 0.125),
  'systems': (1, 0.125)},
 3: {'It': (1, 0.05263),
  'provides': (1, 0.05263),
  'context': (1, 0.05263),
  'concepts': (1, 0.05263),
  'and': (2, 0.10526),
  'key': 

In [13]:
def document_length_norm(doc: str, avg_dl: float, b: float = 0.75):
    dl = len(tokenize(doc))
    return round(1 - b + (b * (dl / avg_dl)), 5)

In [21]:
# calculating bm25 score


def compute_bm25(query: str, k: float = 0.5, b: float = 0.75):
    query_terms: list[str] = query.strip().split(" ")
    query_terms = [q.strip() for q in query_terms if q.strip() in vocab]
    scores = []

    for index, doc in term_frequency.items():
        doc: dict
        score = 0
        for term in query_terms:
            # First part idf's token's idf
            idf, _ = doc_frequency[term]
            # term frequency of term in
            _, tf = doc.get(term, (0,0))
            numerator = tf * (k + 1)
            denominator = tf + (k * document_length_norm(documents[index], avg_dl))
            score += idf * (numerator / denominator)

        scores.append((index, score))

    return scores

In [22]:
scores: list[tuple[int, float]] = compute_bm25("machine learning models")

print(sorted(scores, key=lambda x: x[1], reverse=True))

[(154, 2.301835637317076), (44, 1.9888221492286813), (68, 1.9888221492286813), (72, 1.9888221492286813), (162, 1.9888221492286813), (194, 1.9888221492286813), (32, 1.7315650744653133), (62, 1.7315650744653133), (188, 1.7315650744653133), (98, 0.7977667208459807), (124, 0.7977667208459807), (42, 0.7728912973661021), (100, 0.7728912973661021), (104, 0.7728912973661021), (144, 0.7728912973661021), (12, 0.6838515076260618), (122, 0.6838515076260618), (132, 0.6838515076260618), (136, 0.6838515076260618), (170, 0.6838515076260618), (30, 0.6625281114438891), (52, 0.6625281114438891), (160, 0.6625281114438891), (190, 0.6625281114438891), (33, 0.5975916074628003), (45, 0.5975916074628003), (63, 0.5975916074628003), (69, 0.5975916074628003), (73, 0.5975916074628003), (155, 0.5975916074628003), (163, 0.5975916074628003), (189, 0.5975916074628003), (195, 0.5975916074628003), (24, 0.5908584449302239), (50, 0.5908584449302239), (86, 0.57243469566839), (152, 0.57243469566839), (13, 0.1935055867626517

In [25]:
for score in scores:
    if score[1] > 0.7:
        print(f"Document {score[0]}: {documents[score[0]]}")

Document 32: This document describes machine learning models in real world applications
Document 42: This document discusses transformer models for beginners
Document 44: This document analyzes machine learning models in modern systems
Document 62: This document discusses machine learning models in real world applications
Document 68: This document describes machine learning models with detailed insights
Document 72: This document discusses machine learning models in modern systems
Document 98: This document covers deep learning for beginners
Document 100: This document covers transformer models for beginners
Document 104: This document explores transformer models for beginners
Document 124: This document describes deep learning for beginners
Document 144: This document analyzes transformer models for beginners
Document 154: This document covers machine learning models for beginners
Document 162: This document analyzes machine learning models with practical examples
Document 188: This 

## Message for Reader
- You can follow the structure of this folder to create BM25+ algorithm, BM25L. And other variations of BM25.